In [1]:
import numpy as np

#--------- 加载真实数据 --------------
polar_data_all = np.load("/home/docker/data/private/AuroraData/real_aurora_data_polar/1996/resampled_5min_1996_0405.npy",allow_pickle=True)
polar_timestamps = polar_data_all['utc']
polar_data = np.stack(polar_data_all['aurora_image'], axis=0).astype(np.float32)

# -------- 加载模拟数据 --------------
data1_path = "/home/docker/data/private/AuroraData/generated_aurora_data/1996_omni_aurora/aurora_img_19960401.npy"
data2_path = "/home/docker/data/private/AuroraData/generated_aurora_data/1996_omni_aurora/aurora_img_19960501.npy"
mn_data1 = np.load(data1_path)
mn_data2 = np.load(data2_path)
data_mn_all = np.concatenate((mn_data1, mn_data2), axis=0)
omni_path = "/home/docker/data/private/AuroraData/omni_real_data/omni_5min/1996/omni_19960401_5min.npy"
omni_data = np.load(omni_path)
mn_time = omni_data['utc']

# -------- 加载修复后数据 --------------
repaired_polar_all = np.load("/home/docker/code/Aurora_DDPM/reasult/polar_res/new_res/repaired_polar_unetV3_ckptv2.npy",allow_pickle=True)
repaired_polar = np.stack(repaired_polar_all['image'], axis=0).astype(np.float32)



In [2]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.image import imread
import cartopy.feature as carfeat
import cartopy.io.shapereader as shpreader
from matplotlib.colors import LinearSegmentedColormap
from cartopy.feature.nightshade import Nightshade
import scipy.ndimage
import scipy.interpolate
from datetime import datetime
import os
import math
import aacgmv2
from datetime import timedelta
def convert_datetime64_to_datetime(dt64):
    """将 numpy.datetime64 转换为 datetime.datetime"""
    if dt64 is None:
        return None
    if isinstance(dt64, datetime):
        return dt64
    import pandas as pd
    return pd.Timestamp(dt64).to_pydatetime()

def plot_comparison(timestamp, real_flux, repaired_flux, mn_flux, save_path):
    """
    绘制三张对比图：真实极光、修复极光、预测极光
    """
    timestamp = convert_datetime64_to_datetime(timestamp)
    
    # 自定义极光颜色映射
    colors = [
        (0.0, 0.0, 0.0),
        (0.0, 0.2, 0.0),
        (0.0, 0.5, 0.0),
        (0.0, 0.8, 0.0),
        (0.5, 1.0, 0.0),
        (1.0, 1.0, 0.0),
        (1.0, 0.6, 0.0),
        (1.0, 0.3, 0.0),
        (1.0, 0.0, 0.0),
    ]
    cmap = LinearSegmentedColormap.from_list('aurora', colors, N=256)
    
    # 创建三个数据列表，方便循环处理
    data_list = [real_flux, repaired_flux, mn_flux]
    title_list = ['Real Polar Aurora', 'Repaired Polar Aurora', 'Ovation Aurora']
    
    # 创建世界地图网格
    h, w = 512, 1024
    wx, wy = np.mgrid[-90:90:180 / h, -180:180:360 / w]
    
    # 预先计算所有数据的插值结果
    aimg_list = []
    
    for flux in data_list:
        # 创建极坐标网格
        lat_coords = np.linspace(50, 90, 80)
        mlt_coords = np.linspace(0.0, 24.0, 96)
        mltN, mlatN = np.meshgrid(mlt_coords, lat_coords)
        mlonN_1D_small = aacgmv2.convert_mlt(mltN[0], timestamp, m2a=True)
        mlonN_1D = np.tile(mlonN_1D_small, mlatN.shape[0])
        mlatN_1D = np.squeeze(mlatN.reshape(np.size(mltN), 1))

        # 坐标转换
        (glatN_1D, glonN_1D, galtN) = aacgmv2.convert_latlon_arr(mlatN_1D, mlonN_1D, 100, timestamp,
                                                                 method_code="A2G")

        # 插值到世界地图网格
        geo_2D = np.vstack((glatN_1D, glonN_1D)).T
        fluxN_1D = flux.reshape(7680, 1)

        # 线性插值
        aimg = np.squeeze(scipy.interpolate.griddata(geo_2D, fluxN_1D, (wx, wy), method='linear', fill_value=0))

        # 高斯平滑处理
        aimg = scipy.ndimage.gaussian_filter(aimg, sigma=(2, 3), mode='wrap')
        aimg = aimg.astype(np.float32)
        aimg_list.append(aimg)
    
    # 创建图形 - 3个子图横向排列
    fig = plt.figure(figsize=(28, 12), dpi=150)  # 调整宽度以适应三个子图
    fig.set_facecolor('black')
    
    # 加载背景地图
    background_img = '/home/docker/data/private/AuroraData/background_img/natural-earth-1_large2048px.png'
    map_img = imread(background_img)
    
    # 确定统一的最大值（使用三个数据中的最大值）
    # vmax = max(np.max(aimg_list[0]), np.max(aimg_list[1]), np.max(aimg_list[2]))
    # vmax = max(vmax, 5)  # 确保最小为5，保持一致性
    vmax = 5
    # 绘制三个子图
    for i, (aimg, title) in enumerate(zip(aimg_list, title_list)):
        # 添加子图 - 1行3列
        ax = fig.add_subplot(1, 3, i+1, projection=ccrs.Orthographic(116.2, 90))
        
        # 显示背景地图
        ax.imshow(map_img, origin='upper', transform=ccrs.PlateCarree(),
                  extent=[-180, 180, -90, 90], zorder=0)
        
        # 添加网格线
        gl = ax.gridlines(linestyle='solid', alpha=0.5, color='white')
        gl.n_steps = 100
        gl.xlocator = matplotlib.ticker.FixedLocator(np.arange(-180, 190, 45))
        gl.ylocator = matplotlib.ticker.FixedLocator(np.arange(-90, 100, 10))
        
        # 添加海岸线
        ax.coastlines('10m', color='white', alpha=0.4)
        
        # 添加夜晚阴影
        ax.add_feature(Nightshade(timestamp))
        
        # 显示极光图像
        img = ax.imshow(aimg,
                        vmin=0,
                        vmax=vmax,
                        transform=ccrs.PlateCarree(),
                        extent=[-180, 180, -90, 90],
                        origin='lower',
                        zorder=3,
                        alpha=0.8,
                        cmap=cmap)
        
        ax.set_facecolor('black')
        
        # 设置标题
        ax.set_title(title, color='white', fontsize=18, fontweight='bold', pad=20)
    
    # 添加总标题
    time_str = timestamp.strftime("%Y-%m-%d %H:%M UT")
    fig.suptitle(f'Aurora Comparison - {time_str}', 
                 color='white', fontsize=22, fontweight='bold', y=0.95)
    
    # 添加共享的颜色条（放在底部中央）
    cbar_ax = fig.add_axes([0.35, 0.05, 0.3, 0.02])  # [left, bottom, width, height]
    cbar = plt.colorbar(img, cax=cbar_ax, orientation='horizontal')
    cbar.set_alpha(1)
    cbar.ax.tick_params(labelsize=14, colors='white')
    cbar.set_label(r'Aurora Flux $\mathrm{erg\/cm^{-2}\/s^{-1}}$',
                   color='white', fontsize=16)
    
    plt.tight_layout(rect=[0, 0.1, 1, 0.95])  # 为颜色条留出空间
    plt.savefig(save_path, dpi=150, facecolor='black', bbox_inches='tight')
    plt.close(fig)
    print(f"已保存三合一对比图: {save_path}")
# 调用示例
# plot_comparison(timestamp, real_flux, repaired_flux, mn_flux, save_path)

In [ ]:
import pandas as pd
save_dir = "/home/docker/code/Aurora_DDPM/reasult/polar_res/new_res/comparison_images_unetv3_ckptv2/"
os.makedirs(save_dir, exist_ok=True)
for i in range(0, 400):
    time1 = polar_timestamps[i].astype('datetime64[s]')
    time1 = pd.Timestamp(time1).to_pydatetime()
    base_time = datetime.fromisoformat("1996-04-01T00:00:00")
    delta = time1 - base_time
    idx_mn = int(delta.total_seconds() // 300) # Assuming 5-minute intervals
    mn_data = data_mn_all[idx_mn]
    save_path = os.path.join(save_dir, f"comparison_{i:03d}.png")
    plot_comparison(time1, polar_data[i], repaired_polar[i], data_mn_all[idx_mn], save_path)